# **Correlation Analysis of Cell Intensities**

This notebook is designed to analyze the correlation of cell intensities over time.

### **Imports and Setup**

We start by importing necessary libraries:

In [ ]:
import os
import tifffile
import pickle
import random
import pandas as pd
import napari
import numpy as np
import matplotlib.pyplot as plt
seg_utils = __import__('0_cct_utils')

### **Define Paths**

This section is for inputting the paths to the pkls, tifs, and masks that you want to analyze. To generate the masks and `.pkl` files needed as input here, run the scripts:
* `1_generate_masks.ipynb` and
* `2_tracking_refinement.ipynb`, respectively.

In [ ]:
# Get the parent directory of the current working directory
parent_dir = os.path.abspath(os.path.join(os.getcwd(), os.pardir))  # Moves up one level

# Define model name and base file name separately
model_folder = "ModelAB1"
file_name = 'CON_220525_cluster1_20min_10i_340ms'

# Construct paths dynamically
raw_path = os.path.normpath(os.path.join(parent_dir, "raw_data", model_folder))
mask_path = os.path.normpath(os.path.join(parent_dir, "masks_tracked", model_folder))
pkl_path = os.path.normpath(os.path.join(parent_dir, "pkl_data", model_folder))
export_path = os.path.normpath(os.path.join(parent_dir, "plots"))
latex_path = os.path.normpath(os.path.join(parent_dir, "plots", "latex"))

# Dynamically construct full paths using file_name
raw_file_path = os.path.join(raw_path, f'{file_name}.tif')
mask_file_path = os.path.join(mask_path, f'{file_name}.tif')

# Set a modified list of pkls if needed
new_pkls = [f'{file_name}.pkl']
pkls = sorted(new_pkls)

### **Load `.pkl` File**

We load the pkl file containing the cell tracking data.

In [ ]:
#Load pkl file
for pkl in sorted(pkls):
    file_path = os.path.join(pkl_path, pkl)
    if os.path.exists(file_path):
        with open(file_path, 'rb') as f:
            d = pickle.load(f)
        print('Loaded ' + str(pkl))
    else:
        print(f"File not found: {file_path}")

### **Load Raw Image and Masks**

We load the raw image and corresponding mask file, and check if their dimensions match.

In [ ]:
# Load raw image and masks
try:
    # Load the TIFF image (raw fluorescence image)
    X = tifffile.imread(raw_file_path)
    print(f"Raw image shape: {X.shape}")
    
    # Load the corresponding mask file (segmented cell masks)
    Y = tifffile.imread(mask_file_path)
    print(f"Mask image shape: {Y.shape}")
    
    # Check if mask image and raw image match in size
    if X.shape[:2] != Y.shape[:2]:
        raise ValueError("The raw image and mask image dimensions do not match.")
    
    # Get the labels for the tracked cells (excluding the last cell)
    tracked_cells = list(d.keys())[:-1]  # Assuming 'd' contains tracked cell data
    print(f"Tracked cells: {tracked_cells}")

    # Get the unique labels in the mask image (these correspond to the different cell regions)
    unique_labels = np.unique(Y)

    # Filter the mask to keep only the regions corresponding to tracked cells
    filtered_mask = np.zeros_like(Y)  # Initialize an empty mask with the same shape as Y
    for cell_id in tracked_cells:
        if cell_id in unique_labels:
            filtered_mask[Y == cell_id] = cell_id  # Keep only the regions of tracked cells
        else:
            print(f"Warning: Cell {cell_id} not found in the mask.")

    # Create a Napari viewer
    viewer = napari.Viewer()

    # Add the raw image to the viewer
    viewer.add_image(X, name='Raw Image', colormap='gray')

    # Add the filtered mask to the viewer
    viewer.add_labels(filtered_mask, name='Filtered Masks', opacity=0.35)

    # Start the Napari viewer (this opens a GUI window for interactive visualization)
    napari.run()

except FileNotFoundError as e:
    print(f"Error loading file: {e}")
except ValueError as e:
    print(f"Error with image dimensions: {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")



### **Define Analysis Parameters**

Here you can input your specific parameters for the analysis.

In [ ]:
#Here you can input your specific parameters for the analysis
#TODO: Save correlation plots for each pkl file and for window size 25
#TODO: Save the filtering plots for each pkl file and for window size 25

frame_rate = 0.2    #Frame rate of your time series (frames per second)
window_size = 25    #Window size for noise reduction in the signals (number of frames)
max_lag = 200       #Maximum time step for correlation analysis (number of frames), set based on the maximum period of oscillation observed for the smoothed signal
min_overlap = 500   # Minimum number of overlapping points for correlation calculation

plot_name =  f"{file_name}_{window_size}.pdf"               #Name of the .png file to save the plot as, change end depending on window size chosen
plot_file_path = os.path.join(export_path, plot_name)       #Full path to the plot file

### **Preprocess Intensity Data**

We normalize the intensities, apply smoothing, and plot the normalized raw and filtered intensities for a random cell.

In [ ]:
# Preprocessing of the intensity data
# Step 1: Normalize intensities per cell
intensities_normalized = {}
for cell_id in tracked_cells:
    if cell_id in d:
        intensities = d[cell_id]['intensities']
        max_intensity = np.max(intensities)
        normalized_intensities = intensities / max_intensity if max_intensity > 0 else intensities
        intensities_normalized[cell_id] = normalized_intensities

# Step 2: Apply smoothing (replace local function with seg_utils if available)
filtered_intensities = seg_utils.apply_smoothing_to_normalized(intensities_normalized, window_size, smoothing_method='padding')

# Step 3: Plot normalized and smoothed intensities for a random cell
random_cell_id = random.choice(list(intensities_normalized.keys()))
normalized_raw_intensities = intensities_normalized[random_cell_id]
filtered_intensity = filtered_intensities[random_cell_id]
time_axis = np.arange(len(normalized_raw_intensities))
if len(filtered_intensity) != len(time_axis):
    time_axis = np.arange(len(filtered_intensity))

plt.figure(figsize=(10, 6))
plt.plot(time_axis, normalized_raw_intensities[:len(time_axis)], label='Normalized Raw Intensities', color='blue', alpha=0.6)
plt.plot(time_axis, filtered_intensity, label='Filtered Intensities (Rolling Window)', color='red')
plt.title(f'Normalized Intensity Curves for Cell {random_cell_id}')
plt.xlabel('Time (frames)')
plt.ylabel('Normalized Intensity')
plt.legend()
plt.grid(True)

smoothing_file_name = f"{file_name}_{window_size}_normalized_intensity_curves.pdf"
smoothing_image_path = os.path.join(export_path, smoothing_file_name)
plt.savefig(smoothing_image_path, bbox_inches='tight', dpi=300)
plt.show()

### **Calculate Correlation Values**

This section calculates the correlation values between all cell pairs for the filtered intensities and lags up until the given max_lag. It also finds all the maximum correlation values for each cell pair and saves them separately.

In [ ]:
# This section calculates the correlation values between all cell pairs for the filtered intensities and lags up until max_lag.
# It also finds all the maximum correlation values for each cell pair and saves them separately.

# Generate all possible pairs of cells for use in cross-correlation analysis
all_possible_pairs = [(cell1, cell2) for i, cell1 in enumerate(tracked_cells) for cell2 in tracked_cells[i+1:]]

# Initialize lists to store correlation data
all_correlations_filtered = []
max_correlations_filtered = []

# Calculate correlations for each pair
for cell1, cell2 in all_possible_pairs:
    if cell1 not in filtered_intensities or cell2 not in filtered_intensities:
        print(f"Warning: Missing baseline-subtracted intensity data for cells {cell1} or {cell2}. Skipping this pair.")
        continue

    # Use the seg_utils function instead of local get_cc
    correlation_data_filtered = seg_utils.calculate_cross_correlation(
        cell1, cell2, max_lag, filtered_intensities, min_overlap=min_overlap
    )
    
    all_correlations_filtered.append(((cell1, cell2), correlation_data_filtered))
    max_corr_filtered = np.nanmax(correlation_data_filtered['correlations'])  # Skip NaN values
    max_correlations_filtered.append(((cell1, cell2), max_corr_filtered))

# Print the correlation values for all cell pairs from max_correlations_filtered, sorted by correlation value
print("\nAll Cell Pairs and Their Maximum Correlations:")

# Sort and print maximum correlations
max_correlations_filtered_sorted = sorted(max_correlations_filtered, key=lambda x: x[1], reverse=True)
for (cell1, cell2), max_corr in max_correlations_filtered_sorted:
    print(f"Cells ({cell1}, {cell2}) - Maximum Correlation: {max_corr}")

### **Calculate Maximum and Minimum Correlation Values**

This section calculates the maximum and minimum correlation values for each cell pair and saves them in a table for visualization.

In [ ]:
# This section calculates the maximum and minimum correlation values for each cell pair and saves them in a table for visualization

# Generate all possible pairs of cells for use in cross-correlation analysis
all_possible_pairs = [(cell1, cell2) for i, cell1 in enumerate(tracked_cells) for cell2 in tracked_cells[i+1:]]

# Initialize lists to store correlation data
all_correlations_filtered = []
max_correlations_filtered = []
min_correlations_filtered = []

# Set the distance threshold for filtering (in pixels)
distance_threshold = 100

# Compute correlations only for spatially close pairs
for cell1, cell2 in all_possible_pairs:
    if not seg_utils.filter_cell_pairs_by_distance(cell1, cell2, d, threshold=distance_threshold):
        continue  # Skip distant pairs

    if cell1 not in filtered_intensities or cell2 not in filtered_intensities:
        continue

    # Calculate cross-correlation for the pair
    correlation_data_filtered = seg_utils.calculate_cross_correlation(cell1, cell2, max_lag, filtered_intensities, min_overlap=min_overlap)
    
    all_correlations_filtered.append(((cell1, cell2), correlation_data_filtered))
    
    max_corr_filtered = np.nanmax(correlation_data_filtered['correlations'])  # Skip NaNs
    min_corr_filtered = np.nanmin(correlation_data_filtered['correlations'])
    
    max_correlations_filtered.append(((cell1, cell2), max_corr_filtered))
    min_correlations_filtered.append(((cell1, cell2), min_corr_filtered))

print(f"Computed filtered correlations for {len(max_correlations_filtered)} spatially close cell pairs.")

# Print the correlation values for all cell pairs: maximum, minimum
print("\nAll Cell Pairs and Their Maximum and Minimum Correlations:")

# Sort and print correlations
max_correlations_filtered_sorted = sorted(max_correlations_filtered, key=lambda x: x[1], reverse=True)
min_correlations_filtered_sorted = sorted(min_correlations_filtered, key=lambda x: x[1])

print("\nMaximum Correlations:")
for (cell1, cell2), max_corr in max_correlations_filtered_sorted:
    print(f"Cells ({cell1}, {cell2}) - Maximum Correlation: {max_corr}")

print("\nMinimum Correlations:")
for (cell1, cell2), min_corr in min_correlations_filtered_sorted:
    print(f"Cells ({cell1}, {cell2}) - Minimum Correlation: {min_corr}")

# Prepare tables of top 10 and bottom 10 correlations
top_10_pairs = max_correlations_filtered_sorted[:10]
bottom_10_pairs = max_correlations_filtered_sorted[-10:]

# Create and save the table for the top 10 and bottom 10 correlations
# seg_utils.create_and_save_table(top_10_pairs, "Top 10 Highest Correlations", file_name)
# seg_utils.create_and_save_table(bottom_10_pairs, "Bottom 10 Lowest Correlations", file_name)

# Generate and save LaTeX tables for top 10 and bottom 10 correlations
seg_utils.generate_latex_table(top_10_pairs, "Top 10 Highest Correlations", f"{file_name}_correlationTableTop10", file_name, latex_path, min_correlations_filtered)
seg_utils.generate_latex_table(bottom_10_pairs, "Bottom 10 Lowest Correlations", f"{file_name}_correlationTableBottom10", file_name, latex_path, min_correlations_filtered)

### **Find Significant Correlation Threshold**

This section finds the significant threshold for the correlation values by shuffling the intensities and comparing the observed correlation values to the null distribution. This threshold is then used in the rest of the analysis to find significant correlations.

In [ ]:
# --- Generate filtered pairs based on distance threshold ---
filtered_possible_pairs = [
    (cell1, cell2)
    for cell1, cell2 in all_possible_pairs
    if seg_utils.filter_cell_pairs_by_distance(cell1, cell2, d, threshold=distance_threshold)
]

print(f"Total filtered pairs (within {distance_threshold}px): {len(filtered_possible_pairs)}")

# --- Sort observed max correlations for threshold computation ---
max_correlations_filtered_sorted_asc = sorted(max_correlations_filtered, key=lambda x: x[1])

# --- Compute the null distribution with filtered pairs ---
null_distribution = seg_utils.build_null_distribution(filtered_possible_pairs, filtered_intensities, max_lag, num_permutations=1000, min_overlap=min_overlap)

# --- Find significant threshold ---
significant_threshold, significant_p_value = seg_utils.get_significant_correlation_threshold(
    max_correlations_filtered_sorted_asc,
    null_distribution,
    p_value_threshold=0.05
)

# --- Set the threshold for use in analysis ---
threshold = significant_threshold

### **Plot Correlation Curves and Shifted Intensity Curves**

This section plots the correlation curves and the shifted intensity curves for cell pairs with correlation values above the significant threshold calculated in the previous section.<br>
The time lag for shifting the intensity curves in relation to each other is determined by the time lag for the maximum correlation value.

In [ ]:
threshold = significant_threshold  # Use the significant threshold for filtering 

# Get the list of max correlations above the threshold
correlations_above_threshold = [
    (pair, corr) for pair, corr in max_correlations_filtered if corr > threshold
]

# Sort by the correlation value (descending order)
sorted_correlations_above_threshold = sorted(correlations_above_threshold, key=lambda x: x[1], reverse=True)

# Plot functions
def shift_intensity_curve(intensities, time_lag):
    num_frames = len(intensities)
    shifted_intensity = np.full(num_frames, np.nan)
    if time_lag > 0:
        shifted_intensity[time_lag:] = intensities[:-time_lag]
    elif time_lag < 0:
        shifted_intensity[:time_lag] = intensities[-time_lag:]
    else:
        shifted_intensity = intensities.copy()
    return shifted_intensity

def plot_correlation_curve(correlation_data, frame_rate, cell1, cell2, max_lag):
    time_lags_in_seconds = correlation_data['time_lags'] * (1 / frame_rate)
    plt.figure(figsize=(12, 6))
    plt.plot(time_lags_in_seconds, correlation_data['correlations'],
             label=f'Correlation for Cell {cell1} vs Cell {cell2}', color='green')
    plt.title(f'Cross-Correlation Curve for Cells {cell1} and {cell2}')
    plt.xlabel('Time Lag (seconds)')
    plt.ylabel('Cross-Correlation')
    plt.grid(True)
    plt.legend()
    plt.show()

def plot_intensity_curves(cell1, cell2, correlation_data, frame_rate, intensities_filtered):
    intensities1 = intensities_filtered[cell1]
    intensities2 = intensities_filtered[cell2]
    max_corr_index = np.nanargmax(correlation_data['correlations'])
    max_time_lag = correlation_data['time_lags'][max_corr_index]
    shifted_intensities2 = shift_intensity_curve(intensities2, max_time_lag)

    if max_time_lag > 0:
        start_idx, end_idx = max_time_lag, len(intensities1)
    elif max_time_lag < 0:
        start_idx, end_idx = 0, len(intensities1) + max_time_lag
    else:
        start_idx, end_idx = 0, len(intensities1)

    valid_intensities1 = intensities1[start_idx:end_idx]
    valid_intensities2 = shifted_intensities2[start_idx:end_idx]
    valid_time_axis = np.arange(start_idx, end_idx) / frame_rate / 60

    plt.figure(figsize=(12, 6))
    plt.plot(valid_time_axis, valid_intensities1, label=f'Cell {cell1} Intensity', color='blue')
    plt.plot(valid_time_axis, valid_intensities2, label=f'Cell {cell2} Intensity (shifted)', color='orange')
    plt.title(f'Normalized Intensities for Cells {cell1} and {cell2} (Time Lag = {max_time_lag} frames)')
    plt.xlabel('Time (minutes)')
    plt.ylabel('Normalized Intensity')
    plt.legend()
    plt.grid(True)
    plt.show()

# Plotting for all pairs with correlations above the threshold
for (cell1, cell2), _ in sorted_correlations_above_threshold:
    correlation_data_filtered = next(
        cor_data for (c1, c2), cor_data in all_correlations_filtered
        if (c1 == cell1 and c2 == cell2) or (c1 == cell2 and c2 == cell1)
    )
    plot_correlation_curve(correlation_data_filtered, frame_rate, cell1, cell2, max_lag)
    plot_intensity_curves(cell1, cell2, correlation_data_filtered, frame_rate, filtered_intensities)


### **Visualize Network and Non-Network Cells**

This section visualizes the network of cell correlations and the center of mass (COM) of the tracked cells at a specific frame using Napari. The network cells are those with correlations above the significant threshold, while the non-network cells have correlations below the threshold.

In [ ]:
# This is the frame that will later be used for visualizing the COMs of the filtered cell pairs
frame_to_use = 720  

# You can adjust this value to control how far to search for valid COMs in neighboring frames if some COMs are missing valid coordinates in the chosen frame
max_frame_gap = 250 

# Threshold for correlation values to be considered (normalized values between 0 and 1)
threshold = significant_threshold

# Get the list of max correlations above the threshold
correlations_above_threshold = [
    (pair, corr) for pair, corr in max_correlations_filtered if corr > threshold
]

# Extract the labels from the cell pairs with correlations above the threshold
filtered_pairs = [pair for pair, _ in correlations_above_threshold]
print(f"Filtered pairs (above threshold): {filtered_pairs}")

# Extract unique cell labels from the filtered pairs
unique_labels_filtered = seg_utils.extract_unique_labels_from_pairs(filtered_pairs)

# Identify non-network cells by excluding those present in the network labels
non_network_cells = list(set(tracked_cells) - set(unique_labels_filtered))
print(f"Non-network cells (after exclusion from network labels): {non_network_cells}")

# Calculate and store the center of mass for each unique cell at the specified frame for network cells
filtered_cell_coms_at_frame = {
    cell: tuple(seg_utils.calculate_valid_center_of_mass(Y, cell, frame_to_use, max_frame_gap))
    for cell in unique_labels_filtered
    if seg_utils.is_valid_com(seg_utils.calculate_valid_center_of_mass(Y, cell, frame_to_use, max_frame_gap))
}

# Calculate and store the center of mass for each unique cell at the specified frame for non-network cells
non_network_cell_coms_at_frame = seg_utils.calculate_and_save_cell_coms_at_frame(Y, non_network_cells, frame_to_use, max_frame_gap)


### **Calculate Center of Mass for Network and Non-Network Cells**

This section calculates the center of mass (COM) for each cell at a specific frame.<br>
If the COM is missing or invalid, it searches for valid coordinates in neighboring frames. The COMs are then used to visualize the network and non-network cells.

In [ ]:
#This section creates the Napari image of the network of cell correlations and the COMs of the tracked cells at a specific frame and saves them to the path specified in the export_path variable
#under the name specified in the plot_name variable

seg_utils.plot_network_and_cell_coms_in_napari(
    X, correlations_above_threshold, filtered_cell_coms_at_frame,
    non_network_cell_coms_at_frame, frame_to_use, export_path=export_path, plot_file_path=plot_file_path
)